# 02 — Línea base y cierres

**CC2017 Modelación y Simulación — Ciclo 2, 2026**

Este notebook hace tres cosas:

1. Fija un conjunto de **orígenes y destinos** reproducible y mide la matriz
   origen-destino de referencia sobre la red abierta.
2. Genera los **archivos de cierre** que `osrm-customize` consume, y los aplica.
3. **Verifica que el cierre efectivamente cerró.** Esto no es un trámite: es el
   punto donde un proyecto así falla en silencio — el CSV se escribe, OSRM lo
   acepta sin quejarse, y las rutas no cambian porque los IDs no correspondían.
   Las tres comprobaciones de
   [`estructura-proyecto.md` §12](../estructura-proyecto.md) van como
   aserciones.

Al final sale un **Δ% preliminar** por par de zonas. Preliminar porque todavía
no hay tráfico: son tiempos de flujo libre sobre la red abierta contra la
cerrada, sin congestión y sin variabilidad de conductor. Eso llega en el
notebook 04. Lo de aquí es el piso: el desvío puramente geométrico que impone
la red.

## 0. Configuración

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import osrm
import red

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)

rng = np.random.default_rng(config.SEMILLA)

LOCAL_OK = osrm.disponible(config.OSRM_LOCAL)
SERVIDOR = config.OSRM_LOCAL if LOCAL_OK else config.OSRM_PUBLICO

print(f"servidor: {SERVIDOR}")
if not LOCAL_OK:
    print("\nOSRM local no responde. La línea base sí se puede medir contra el")
    print("servidor público, pero los CIERRES (§4 en adelante) necesitan el local:")
    print("  ./osrm/construir.sh")
    print("  docker compose -f osrm/docker-compose.yml up -d")

servidor: http://127.0.0.1:5001


## 1. Orígenes y destinos

Se sortean `N_PUNTOS_POR_ZONA` puntos uniformes dentro de un disco alrededor del
centro de cada zona y se **anclan a la vía más cercana** con el servicio
`nearest` de OSRM. Sin el anclaje, un origen podría caer en el techo de un
edificio o dentro de un barranco, y OSRM lo engancharía a la calle que se le
antoje.

El sorteo usa la semilla global, así que este conjunto de puntos es el mismo en
todas las corridas y en todas las máquinas. Es deliberado: los escenarios
abierto y cerrado tienen que compararse sobre exactamente los mismos pares
origen-destino, o el Δ% no significa nada
([§5.6, números aleatorios comunes](../estructura-proyecto.md)).

In [2]:
def sortear_en_disco(centro, radio_m, n, rng):
    # Puntos uniformes en el disco. La raíz de u es lo que evita que se
    # apelotonen en el centro: sin ella la densidad crecería hacia adentro.
    lat0, lon0 = centro
    u, theta = rng.random(n), rng.random(n) * 2 * np.pi
    r = radio_m * np.sqrt(u)
    dlat = (r * np.cos(theta)) / 111_320
    dlon = (r * np.sin(theta)) / (111_320 * np.cos(np.radians(lat0)))
    return list(zip(lat0 + dlat, lon0 + dlon))


puntos = []
for clave, z in config.ZONAS.items():
    crudos = sortear_en_disco(z["centro"], config.RADIO_ZONA_M,
                              config.N_PUNTOS_POR_ZONA, rng)
    for i, p in enumerate(crudos):
        anc = osrm.nearest(p, servidor=SERVIDOR)
        puntos.append({
            "id": f"{clave}_{i}",
            "zona": clave,
            "lat": anc["lat"], "lon": anc["lon"],
            "calle": anc["nombre"] or "(sin nombre)",
            "ajuste_m": anc["distancia_m"],
        })

od = pd.DataFrame(puntos)
display(od.head(10).round(5))
print(f"\n{len(od)} puntos · {len(od) * (len(od) - 1)} pares ordenados")
print(f"Ajuste al anclar: mediana {od['ajuste_m'].median():.0f} m, "
      f"máximo {od['ajuste_m'].max():.0f} m")

assert od["ajuste_m"].max() < 500, "algún punto quedó demasiado lejos de una calle"
COORDS = list(zip(od["lat"], od["lon"]))

,id,zona,lat,lon,calle,ajuste_m
0,z10_0,z10,14.60534,-90.51110,5a Avenida,31.87529
1,z10_1,z10,14.59683,-90.51019,7a Avenida,25.35649
2,z10_2,z10,14.59571,-90.50527,12 Avenida,18.03746
3,z10_3,z10,14.60125,-90.51735,(sin nombre),11.59178
4,z10_4,z10,14.59242,-90.50897,17 Calle,2.17018
5,z10_5,z10,14.60282,-90.50608,(sin nombre),3.44572
6,z10_6,z10,14.59145,-90.50961,(sin nombre),6.75642
7,z10_7,z10,14.59350,-90.51545,(sin nombre),4.56516
8,z15_0,z15,14.59748,-90.49167,18 Avenida B,8.00182
9,z15_1,z15,14.61201,-90.48611,(sin nombre),13.21156



24 puntos · 552 pares ordenados
Ajuste al anclar: mediana 16 m, máximo 105 m


## 2. Matriz origen-destino de referencia

El servicio `table` de OSRM devuelve la matriz completa en **una sola petición**,
en vez de las 552 llamadas a `route` que haría falta si no. Sobre el servidor
local no hay tope práctico; el público sí limita a 100 coordenadas.

In [3]:
def matriz(servidor):
    t = osrm.tabla(COORDS, servidor=servidor)
    dur = np.array(t["duraciones_s"], dtype=float)
    dis = np.array(t["distancias_m"], dtype=float)
    np.fill_diagonal(dur, np.nan)   # el viaje de un punto a sí mismo no existe
    np.fill_diagonal(dis, np.nan)
    return dur, dis


dur_base, dis_base = matriz(SERVIDOR)

print(f"matriz {dur_base.shape}")
print(f"pares sin ruta: {int(np.isnan(dur_base).sum() - len(COORDS))}")
print(f"duración de flujo libre — mediana {np.nanmedian(dur_base) / 60:.1f} min, "
      f"p95 {np.nanpercentile(dur_base, 95) / 60:.1f} min")
print("\nRecordatorio: son tiempos de FLUJO LIBRE. No son predicciones de viaje real.")

matriz (24, 24)
pares sin ruta: 0
duración de flujo libre — mediana 7.7 min, p95 14.2 min

Recordatorio: son tiempos de FLUJO LIBRE. No son predicciones de viaje real.


In [4]:
# Resumen por par de zonas: el nivel al que se reportan los resultados.
zonas_idx = od["zona"].values


def por_zonas(m, agg=np.nanmedian):
    claves = list(config.ZONAS)
    out = pd.DataFrame(index=claves, columns=claves, dtype=float)
    for zo in claves:
        for zd in claves:
            bloque = m[np.ix_(zonas_idx == zo, zonas_idx == zd)]
            out.loc[zo, zd] = agg(bloque) if not np.all(np.isnan(bloque)) else np.nan
    return out


base_zonas = por_zonas(dur_base) / 60
print("Mediana de duración en flujo libre por par de zonas (minutos)\n"
      "filas = origen, columnas = destino")
display(base_zonas.round(2))

Mediana de duración en flujo libre por par de zonas (minutos)
filas = origen, columnas = destino


,z10,z15,z16
z10,3.65,10.34,11.20
z15,11.88,4.31,3.96
z16,13.10,4.92,4.97


## 3. Rutas con nodos

`table` da tiempos pero no dice **por dónde** pasa cada ruta. Para verificar el
cierre hacen falta los IDs de nodo OSM que la ruta atraviesa, y eso solo lo da
`route` con `annotations=nodes`.

Se toma una muestra de pares en vez de los 552, porque para la verificación no
hace falta más y contra el servidor público las peticiones se cuentan.

In [5]:
N_MUESTRA = 60
rng_m = np.random.default_rng(config.SEMILLA + 1)

todos = [(i, j) for i in range(len(COORDS)) for j in range(len(COORDS)) if i != j]
sel = [todos[k] for k in rng_m.choice(len(todos), size=N_MUESTRA, replace=False)]


def rutas_con_nodos(servidor, pares):
    out = {}
    for i, j in pares:
        r = osrm.ruta(COORDS[i], COORDS[j], servidor=servidor, nodos=True)
        out[(i, j)] = r     # None cuando no hay ruta: es un resultado, no un error
    return out


rutas_base = rutas_con_nodos(SERVIDOR, sel)
sin_ruta = sum(1 for r in rutas_base.values() if r is None)
print(f"{len(sel)} rutas · sin ruta: {sin_ruta}")
assert sin_ruta == 0, "con la red abierta no debería faltar ninguna ruta"

largo = np.array([len(r["nodos"]) for r in rutas_base.values()])
print(f"nodos por ruta: mediana {np.median(largo):.0f}, máximo {largo.max()}")

60 rutas · sin ruta: 0
nodos por ruta: mediana 216, máximo 412


## 4. Archivos de cierre

`osrm-customize --segment-speed-file` consume un CSV de
`nodo_origen,nodo_destino,velocidad_kmh`. Velocidad **0** deja el segmento
intransitable.

El CSV se arma pidiendo a Overpass los ways cuyo `name` coincide exactamente
con el de la vía, tomando sus nodos consecutivos y escribiendo los pares **en
ambos sentidos**. Escribir los dos sentidos aunque el way sea `oneway` no
estorba —el sentido que no existe en el grafo se ignora— y ahorra tener que
razonar sobre la dirección del mapeo, que es una fuente clásica de cierres a
medias.

In [6]:
cierres = {}
for clave in ("vista_hermosa", "reforma"):
    esc = config.ESCENARIOS[clave]
    salida = config.DERIVADOS / f"cierre_{clave}.csv"
    cierres[clave] = red.archivo_velocidades(esc["nombres_osm"], salida, velocidad=0)

# Conjuntos cerrados por escenario. Se guardan los DOS, porque no sirven para
# lo mismo: los segmentos son el criterio de verificación (§6) y los nodos solo
# sirven para saber qué intersecciones toca la calle.
nodos_cerrados, segmentos_cerrados = {}, {}
for clave in cierres:
    ws = red.ways_con_nodos(config.ESCENARIOS[clave]["nombres_osm"])
    nodos_cerrados[clave] = set(n for w in ws for n in w["nodos"])
    segmentos_cerrados[clave] = set(red.pares_de_nodos(ws))
    print(f"{clave}: {len(nodos_cerrados[clave])} nodos, "
          f"{len(segmentos_cerrados[clave])} segmentos dirigidos")

print("\nPrimeras líneas de cierre_vista_hermosa.csv:")
print((config.DERIVADOS / "cierre_vista_hermosa.csv").read_text().split("\n", 3)[:3])

cierre_vista_hermosa.csv: 23 ways -> 408 segmentos a 0 km/h


cierre_reforma.csv: 36 ways -> 162 segmentos a 0 km/h


vista_hermosa: 206 nodos, 408 segmentos dirigidos


reforma: 84 nodos, 162 segmentos dirigidos

Primeras líneas de cierre_vista_hermosa.csv:
['323710474,11712134227,0', '323710670,11826395570,0', '323711515,3831118392,0']


## 5. Aplicar un escenario

`osrm/cerrar.sh` re-ejecuta **solo** `osrm-customize` y reinicia el contenedor.
Tarda segundos porque el grafo pesado ya está compilado desde el notebook 01.

Desde aquí hace falta el servidor local: el público no permite cerrar nada.

In [7]:
def aplicar(clave):
    # Deja el servidor sirviendo el escenario `clave`; 'base' quita los cierres.
    if not LOCAL_OK:
        raise RuntimeError("los cierres requieren el servidor OSRM local")
    cmd = ["./osrm/cerrar.sh"]
    if clave != "base":
        cmd.append(f"data/derivados/cierre_{clave}.csv")
    r = subprocess.run(cmd, cwd=config.RAIZ, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"cerrar.sh falló:\n{r.stdout}\n{r.stderr}")
    osrm.esperar(config.OSRM_LOCAL)
    print(f"escenario activo: {config.ESCENARIOS[clave]['etiqueta']}")

In [8]:
resultados = {}

if LOCAL_OK:
    for clave in ("base", "vista_hermosa", "reforma"):
        aplicar(clave)
        dur, dis = matriz(config.OSRM_LOCAL)
        rutas = rutas_con_nodos(config.OSRM_LOCAL, sel)
        resultados[clave] = {"dur": dur, "dis": dis, "rutas": rutas}
        faltan = int(np.isnan(dur).sum() - len(COORDS))
        print(f"  pares sin ruta: {faltan}   "
              f"mediana: {np.nanmedian(dur) / 60:.2f} min\n")
    aplicar("base")   # dejar el servidor en la línea base
else:
    print("Sin servidor local: esta celda queda pendiente.")
    print("Correr ./osrm/construir.sh y levantar el contenedor, luego reejecutar.")

escenario activo: Sin cierres
  pares sin ruta: 0   mediana: 7.67 min



escenario activo: Cierre Boulevard Vista Hermosa
  pares sin ruta: 0   mediana: 8.98 min



escenario activo: Cierre Avenida Reforma (calzada principal)
  pares sin ruta: 0   mediana: 7.67 min



escenario activo: Sin cierres


## 6. Verificación del cierre

Las tres comprobaciones de [`estructura-proyecto.md` §12](../estructura-proyecto.md).
Van como aserciones porque un cierre que no cierra produce un Δ% pequeño y
plausible, es decir: un error que no se nota.

1. **Nodos ausentes.** Ninguna ruta del escenario cerrado toca un nodo del
   conjunto cerrado.
2. **Monotonía.** Cerrar calles nunca acelera un viaje: `T_cerrado ≥ T_abierto`
   para todo par.
3. **Especificidad.** Los pares cuya ruta base ni tocaba la calle conservan
   exactamente su tiempo. Si cambian, el CSV está cerrando de más.

In [9]:
TOL = 1e-6

if resultados:
    for clave in ("vista_hermosa", "reforma"):
        cerrados = nodos_cerrados[clave]
        segs = segmentos_cerrados[clave]
        base_r = resultados["base"]["rutas"]
        cerr_r = resultados[clave]["rutas"]
        d0, d1 = resultados["base"]["dur"], resultados[clave]["dur"]

        # (1) ninguna ruta del escenario cerrado RECORRE un segmento cerrado
        infractoras = [
            par for par, r in cerr_r.items()
            if r is not None and red.recorre_cerrados(r["nodos"], segs)
        ]
        assert not infractoras, (
            f"{clave}: {len(infractoras)} rutas siguen circulando por la calle cerrada")

        # Cuántas apenas CRUZAN la calle por una intersección: es legítimo y por
        # eso el criterio es el segmento y no el nodo.
        cruzan = [
            par for par, r in cerr_r.items()
            if r is not None and cerrados & set(r["nodos"])
        ]

        # (2) monotonía sobre la matriz completa
        peor = np.nanmin(d1 - d0)
        assert peor >= -TOL, f"{clave}: hay pares que se aceleraron ({peor:.3f} s)"

        # (3) especificidad sobre la muestra con nodos
        tocaban = {p for p, r in base_r.items()
                   if r and red.recorre_cerrados(r["nodos"], segs)}
        intactos = [p for p in sel if p not in tocaban]
        movidos = [
            p for p in intactos
            if base_r[p] and cerr_r[p]
            and abs(base_r[p]["duracion_s"] - cerr_r[p]["duracion_s"]) > 1e-3
        ]

        print(f"{clave}")
        print(f"  (1) rutas que aún circulan por la vía    : {len(infractoras)}  ✓")
        print(f"      rutas que solo la cruzan (legítimo)  : {len(cruzan)}")
        print(f"  (2) mínima diferencia cerrado - base     : {peor:+.3f} s  ✓")
        print(f"  (3) rutas base que sí circulaban por ella: "
              f"{len(tocaban)}/{len(sel)}")
        print(f"      de las que NO pasaban, cambiaron      : {len(movidos)}")
        if movidos:
            print("      (algún cambio es normal: el reenrutamiento del resto del")
            print("       tráfico no altera tiempos de flujo libre, pero sí puede")
            print("       hacerlo un empate de rutas que se rompe distinto)")
        print()
else:
    print("Verificación pendiente: hace falta el servidor local.")

vista_hermosa
  (1) rutas que aún circulan por la vía    : 0  ✓
      rutas que solo la cruzan (legítimo)  : 0
  (2) mínima diferencia cerrado - base     : +0.000 s  ✓
  (3) rutas base que sí circulaban por ella: 31/60
      de las que NO pasaban, cambiaron      : 0

reforma
  (1) rutas que aún circulan por la vía    : 0  ✓
      rutas que solo la cruzan (legítimo)  : 7
  (2) mínima diferencia cerrado - base     : +0.000 s  ✓
  (3) rutas base que sí circulaban por ella: 10/60
      de las que NO pasaban, cambiaron      : 0



## 7. Δ% preliminar

El cambio porcentual de la mediana de duración, por par de zonas. Sin tráfico
todavía: es el **desvío geométrico puro** que impone la red. La simulación del
notebook 04 le suma la congestión, y ahí el efecto crece — la función BPR tiene
exponente 4, así que un desvío que además satura la ruta alterna se paga mucho
más caro que en flujo libre.

In [10]:
if resultados:
    d0 = resultados["base"]["dur"]
    resumen = []
    for clave in ("vista_hermosa", "reforma"):
        d1 = resultados[clave]["dur"]
        delta = 100 * (np.nanmedian(d1) - np.nanmedian(d0)) / np.nanmedian(d0)
        p95 = 100 * (np.nanpercentile(d1, 95) - np.nanpercentile(d0, 95)) \
            / np.nanpercentile(d0, 95)
        total = 100 * (np.nansum(d1) - np.nansum(d0)) / np.nansum(d0)
        sinruta = 100 * (np.isnan(d1).sum() - np.isnan(d0).sum()) / d0.size
        resumen.append({
            "escenario": config.ESCENARIOS[clave]["etiqueta"],
            "Δ% mediana": delta, "Δ% p95": p95,
            "Δ% total red": total, "% pares sin ruta": sinruta,
        })
    display(pd.DataFrame(resumen).round(2))

    print("\nΔ% de la mediana por par de zonas — filas = origen, columnas = destino")
    for clave in ("vista_hermosa", "reforma"):
        m0, m1 = por_zonas(d0), por_zonas(resultados[clave]["dur"])
        print(f"\n{config.ESCENARIOS[clave]['etiqueta']}")
        display((100 * (m1 - m0) / m0).round(1))
else:
    print("Δ% pendiente: hace falta el servidor local.")

,escenario,Δ% mediana,Δ% p95,Δ% total red,% pares sin ruta
0,Cierre Boulevard Vista Hermosa,16.97,36.31,31.61,0.0
1,Cierre Avenida Reforma (calzada principal),0.00,0.71,0.74,0.0



Δ% de la mediana por par de zonas — filas = origen, columnas = destino

Cierre Boulevard Vista Hermosa


,z10,z15,z16
z10,0.0,69.8,43.2
z15,36.2,0.0,0.0
z16,22.8,0.0,0.0



Cierre Avenida Reforma (calzada principal)


,z10,z15,z16
z10,0.0,1.7,2.1
z15,0.8,0.0,0.0
z16,0.8,0.0,0.0


## 8. Derivados

Lo que las pistas B y C reciben de aquí.

In [11]:
derivado = {
    "puntos_od": od.round(6).to_dict("records"),
    "n_pares": len(COORDS) * (len(COORDS) - 1),
    "cierres": cierres,
    "nodos_cerrados": {k: len(v) for k, v in nodos_cerrados.items()},
    "segmentos_cerrados": {k: len(v) for k, v in segmentos_cerrados.items()},
    "servidor_usado": SERVIDOR,
    "linea_base": {
        "mediana_s": float(np.nanmedian(dur_base)),
        "p95_s": float(np.nanpercentile(dur_base, 95)),
        "por_zonas_min": base_zonas.round(3).to_dict(),
    },
}
if resultados:
    derivado["delta_preliminar"] = {
        clave: {
            "mediana_%": float(100 * (np.nanmedian(resultados[clave]["dur"])
                                      - np.nanmedian(resultados["base"]["dur"]))
                               / np.nanmedian(resultados["base"]["dur"])),
        }
        for clave in ("vista_hermosa", "reforma")
    }
    np.save(config.DERIVADOS / "02_duraciones.npy",
            np.stack([resultados[k]["dur"] for k in
                      ("base", "vista_hermosa", "reforma")]))
    print("guardado: 02_duraciones.npy  (3 × N × N, orden base/vista_hermosa/reforma)")

p = red.guardar_derivado(derivado, "02_linea_base.json")
print(f"guardado: {p}")

guardado: 02_duraciones.npy  (3 × N × N, orden base/vista_hermosa/reforma)
guardado: /Users/fabianprado/Documents/projects/2026 - 2 /modelacion/proyecto/data/derivados/02_linea_base.json


---

## Qué queda listo

- Conjunto de orígenes y destinos reproducible, anclado a la red real.
- Matriz origen-destino de referencia sobre la red abierta.
- Archivos de cierre para los dos escenarios, con sus conjuntos de nodos.
- Las tres verificaciones de §12 pasando como aserciones.
- Δ% preliminar de flujo libre, que es el piso contra el que se comparará el
  Δ% con congestión.

**Sigue:**

- `03_generadores.ipynb` (pista B) — generadores de variables aleatorias y su
  validación estadística.
- `04_simulacion.ipynb` (pista C) — el motor: llegadas de Poisson, asignación de
  rutas, congestión BPR y números aleatorios comunes entre escenarios.

La interfaz que este notebook le garantiza a la pista C está en
[`estructura-proyecto.md` §11](../estructura-proyecto.md):

```python
osrm.ruta(origen, destino) -> {"distancia_m", "duracion_s", "nodos"} | None
```